# 📊 CodeLens — Remote Evaluation Pipeline

**Project**: CodeLens — Retrieval-Augmented Code Review Agent with Codebase-Aware Context  
**Purpose**: Offline Evaluation of Retrieval Quality & LLM Review Acceptance Metrics  
**Data Source**: 5 TB Google Drive / Cloud Storage Tier (`datasets/feedback/reactions.jsonl`)

---

### 🎯 Evaluation Goals
1. **Review Acceptance Rate**: Proportion of AI review comments accepted (+1) vs rejected (-1) by engineers.
2. **False Positive Rate**: Percentage of rejected comments classified as noise or inaccurate.
3. **Retrieval Ranking Quality**: Precision@K, Mean Reciprocal Rank (MRR), and NDCG@10 for retrieved context functions.

In [ ]:
# Step 1: Install Analysis Dependencies
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "scikit-learn", "matplotlib"], check=True)


In [ ]:
# Step 2: Mount Google Drive or Locate Feedback File
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

FEEDBACK_PATH = Path("./data/feedback/reactions.jsonl")
DRIVE_FEEDBACK_PATH = Path("/content/drive/MyDrive/codelens-storage/datasets/feedback/reactions.jsonl")

if DRIVE_FEEDBACK_PATH.exists():
    FEEDBACK_PATH = DRIVE_FEEDBACK_PATH
    print(f"Loaded feedback from Google Drive: {FEEDBACK_PATH}")
elif FEEDBACK_PATH.exists():
    print(f"Loaded local feedback: {FEEDBACK_PATH}")
else:
    print("Feedback file not found. Creating sample mock feedback dataset for evaluation demonstration.")
    FEEDBACK_PATH.parent.mkdir(parents=True, exist_ok=True)
    sample_data = [
        {"pr_number": 1, "comment_id": "c1", "repo_full_name": "laxmi2577/code-review-agent", "reaction": "+1", "timestamp": "2026-10-10T10:00:00Z"},
        {"pr_number": 1, "comment_id": "c2", "repo_full_name": "laxmi2577/code-review-agent", "reaction": "+1", "timestamp": "2026-10-10T10:05:00Z"},
        {"pr_number": 2, "comment_id": "c3", "repo_full_name": "laxmi2577/code-review-agent", "reaction": "-1", "timestamp": "2026-10-10T11:00:00Z"},
        {"pr_number": 3, "comment_id": "c4", "repo_full_name": "laxmi2577/code-review-agent", "reaction": "+1", "timestamp": "2026-10-10T12:00:00Z"},
        {"pr_number": 3, "comment_id": "c5", "repo_full_name": "laxmi2577/code-review-agent", "reaction": "+1", "timestamp": "2026-10-10T12:30:00Z"},
    ]
    with open(FEEDBACK_PATH, "w", encoding="utf-8") as f:
        for row in sample_data:
            f.write(json.dumps(row) + "\n")

In [ ]:
# Step 3: Compute Review Quality Metrics
records = []
with open(FEEDBACK_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
total_feedback = len(df)
positive_reactions = (df["reaction"] == "+1").sum()
negative_reactions = (df["reaction"] == "-1").sum()

acceptance_rate = (positive_reactions / total_feedback * 100) if total_feedback > 0 else 0.0
rejection_rate = (negative_reactions / total_feedback * 100) if total_feedback > 0 else 0.0

print("=== CODE REVIEW QUALITY METRICS ===")
print(f"Total Human Reactions Logged: {total_feedback}")
print(f"Accepted Comments (+1):       {positive_reactions} ({acceptance_rate:.1f}%)")
print(f"Rejected Comments (-1):       {negative_reactions} ({rejection_rate:.1f}%)")

In [ ]:
# Step 4: Compute Information Retrieval Metrics (Precision@K, MRR, NDCG@10)
def compute_mrr(rankings):
    """Mean Reciprocal Rank: average of 1 / rank_of_first_relevant_item."""
    rr_list = []
    for ranks in rankings:
        relevant_indices = np.where(ranks == 1)[0]
        if len(relevant_indices) > 0:
            rr_list.append(1.0 / (relevant_indices[0] + 1))
        else:
            rr_list.append(0.0)
    return float(np.mean(rr_list))

def compute_precision_at_k(rankings, k=5):
    """Precision@K: fraction of top-k retrieved items that are relevant."""
    p_at_k = []
    for ranks in rankings:
        top_k = ranks[:k]
        p_at_k.append(np.sum(top_k) / k)
    return float(np.mean(p_at_k))

def compute_ndcg_at_k(rankings, k=10):
    """Normalized Discounted Cumulative Gain at K."""
    ndcg_list = []
    for ranks in rankings:
        top_k = ranks[:k]
        dcg = np.sum(top_k / np.log2(np.arange(2, len(top_k) + 2)))
        ideal = np.sort(ranks)[::-1][:k]
        idcg = np.sum(ideal / np.log2(np.arange(2, len(ideal) + 2)))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(ndcg_list))

# Sample simulated retrieval relevance vectors (1 = relevant context, 0 = non-relevant)
mock_retrieval_runs = np.array([
    [1, 1, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
])

p5 = compute_precision_at_k(mock_retrieval_runs, k=5)
mrr = compute_mrr(mock_retrieval_runs)
ndcg10 = compute_ndcg_at_k(mock_retrieval_runs, k=10)

print("=== CODE RETRIEVAL RANKING METRICS ===")
print(f"Precision@5:  {p5:.3f}")
print(f"MRR:          {mrr:.3f}")
print(f"NDCG@10:      {ndcg10:.3f}")

In [ ]:
# Step 5: Generate and Export Evaluation Summary Report
report = {
    "total_feedback": int(total_feedback),
    "positive_reactions": int(positive_reactions),
    "negative_reactions": int(negative_reactions),
    "acceptance_rate": float(acceptance_rate),
    "precision_at_5": float(p5),
    "mrr": float(mrr),
    "ndcg_at_10": float(ndcg10),
}

report_path = Path("./data/evaluations/evaluation_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"Evaluation report generated and saved to: {report_path.resolve()}")